In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ===== データ読み込み =====
train = pd.read_csv("/home/kazok/kaggle/playground-s6e6/data/train.csv")
test  = pd.read_csv("/home/kazok/kaggle/playground-s6e6/data/test.csv")


In [2]:
# モジュール宣言
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score  # ← 追加
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import OrdinalEncoder
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
#特徴量生成セル
test["u_g"] = test["u"] - test["g"]
test["u_r"] = test["u"] - test["r"]
test["u_i"] = test["u"] - test["i"]
test["g_r"] = test["g"] - test["r"]
test["r_i"] = test["r"] - test["i"]
test["i_z"] = test["i"] - test["z"]

test["u_g_ratio"] = test["u"] / (test["g"] + 1e-6)
test["g_r_ratio"] = test["g"] / (test["r"] + 1e-6)
test["redshift_sq"]   = test["redshift"] ** 2
test["redshift_sqrt"] = np.sqrt(test["redshift"])
test["alpha_x_delta"] = test["alpha"] * test["delta"]

train["u_g"] = train["u"] - train["g"]
train["u_r"] = train["u"] - train["r"]
train["u_i"] = train["u"] - train["i"]
train["g_r"] = train["g"] - train["r"]
train["r_i"] = train["r"] - train["i"]
train["i_z"] = train["i"] - train["z"]

train["u_g_ratio"] = train["u"] / (train["g"] + 1e-6)
train["g_r_ratio"] = train["g"] / (train["r"] + 1e-6)
train["redshift_sq"]   = train["redshift"] ** 2
train["redshift_sqrt"] = np.sqrt(train["redshift"])
train["alpha_x_delta"] = train["alpha"] * train["delta"]

test["alpha_sin"] = np.sin(np.deg2rad(test["alpha"]))
test["alpha_cos"] = np.cos(np.deg2rad(test["alpha"]))
test["redshift_log"] = np.log1p(test["redshift"])

train["alpha_sin"] = np.sin(np.deg2rad(train["alpha"]))
train["alpha_cos"] = np.cos(np.deg2rad(train["alpha"]))
train["redshift_log"] = np.log1p(train["redshift"])

# ===== カテゴリ変数の準備 =====
cat_cols = ["spectral_type", "galaxy_population"]
for col in cat_cols:
    train[col] = train[col].astype("category")
    test[col]  = test[col].astype("category")

for col in ["alpha", "delta", "redshift", "u", "g", "r", "i", "z"]:
    cat_name = f"{col}_cat"
    train[cat_name] = np.floor(train[col]).astype("category")
    test[cat_name]  = np.floor(test[col]).astype("category")
    cat_cols.append(cat_name)

for df in [train, test]:
    df["alpha_delta_combo"] = (np.floor(df["alpha"]).astype(str) + "_" + np.floor(df["delta"]).astype(str)).astype("category")
    df["u_z_combo"] = (
        np.floor(df["u"]).astype(str) + "_" + np.floor(df["z"]).astype(str)
    ).astype("category")

cat_cols += ["alpha_delta_combo", "u_z_combo"]

bin_config ={
    "delta": [100, 500],
    "redshift": [50, 100],
    "alpha": [100],
}
kb_models = {}

for col, bins_list in bin_config.items():
    for n_bins in bins_list:
        bin_name = f"{col}_{n_bins}bin"

        kb = KBinsDiscretizer(
            n_bins= n_bins,
            encode='ordinal',
            strategy='quantile',
            subsample=None
        )

        train[bin_name] = kb.fit_transform(train[[col]]).ravel().astype(int)
        test[bin_name] = kb.transform(test[[col]]).ravel().astype(int)

        train[bin_name] = train[bin_name].astype("category")
        test[bin_name] = test[bin_name].astype("category")

        kb_models[bin_name] = kb
        cat_cols.append(bin_name)
# ===== 目的変数のエンコード =====
le = LabelEncoder()
train["class_encoded"] = le.fit_transform(train["class"])



str_cat_cols = ["spectral_type", "galaxy_population", "alpha_delta_combo", "u_z_combo"]
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train[str_cat_cols] = oe.fit_transform(train[str_cat_cols]).astype(int)
test[str_cat_cols]  = oe.transform(test[str_cat_cols]).astype(int)

# category型に戻す
for col in str_cat_cols:
    train[col] = train[col].astype("category")
    test[col]  = test[col].astype("category")


# ===== 特徴量定義 =====
feature_cols = [
    "alpha", "delta", "u", "g", "r", "i", "z", "redshift",
    "u_g", "g_r", "r_i", "i_z",
    "alpha_sin", "alpha_cos", "redshift_log",
    "spectral_type", "galaxy_population",
    "alpha_cat", "delta_cat", "redshift_cat",
    "u_cat", "g_cat", "r_cat", "i_cat", "z_cat",
    # combo
    "alpha_delta_combo", "u_z_combo",
    "delta_100bin", "delta_500bin",
    "redshift_50bin", "redshift_100bin",
    "alpha_100bin",
]

X = train[feature_cols]
y = train["class_encoded"]
X_test = test[feature_cols]


/home/kazok/miniconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/kazok/miniconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/kazok/miniconda3/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/kazok/miniconda3/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='a

In [4]:


# カスタム評価関数
def xgb_balanced_accuracy(y_pred, dtrain):
    y_true = dtrain.get_label().astype(int)
    y_pred_class = np.argmax(
        y_pred.reshape(-1, 3), axis=1
    )
    score = balanced_accuracy_score(y_true, y_pred_class)
    return "balanced_accuracy", score


In [5]:
#RealMLP　利用クラス宣言


# ── Preprocessing ─────────────────────────────────────────────────────────────
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms
                      if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X: np.ndarray, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X: np.ndarray, y=None) -> np.ndarray:
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X


# ── Model components ──────────────────────────────────────────────────────────
class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []

        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList([nn.Embedding(dim, embed_dim) for _ in range(n_ens)])
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        batch_size, n_ens, _ = x.shape
        features = []
        if self.onehot_features:
            onehot_x    = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh    = sum(onehot_dims)
            encoded     = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx : idx + 1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)
        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx : feat_idx + 1].long()
                feat_embs.append(emb_list[model_idx](indices))
            feat_combined = torch.cat(feat_embs, dim=1)
            features.append(feat_combined)
        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias   = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4,
                 freq_scale=0.1, activation=nn.GELU):
        super().__init__()
        self.n_ens      = n_ens
        self.n_features = n_features
        self.out_dim    = out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim))
        self.b2 = nn.Parameter(torch.zeros(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        periodic = torch.cos(
            2 * math.pi * (x.unsqueeze(-1) * self.w1.unsqueeze(0) + self.b1.unsqueeze(0))
        )
        transformed = self.act(
            torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2) + self.b2.unsqueeze(0)
        )
        feat = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return feat.flatten(start_dim=2)


class RealMLP(nn.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg):
        super().__init__()
        n_ens     = cfg["n_ens"]
        embed_dim = cfg["embed_dim"]
        self.n_ens = n_ens

        self.cate = CategoricalFeatureLayer(
            n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim,
            onehot_thresh=cfg["onehot_thresh"],
        )
        self.num_embed = PBLDEmbedding(
            n_ens=n_ens, n_features=n_numerical,
            hidden_dim=cfg["pbld_hidden_dim"], out_dim=cfg["pbld_out_dim"],
            freq_scale=cfg["pbld_freq_scale"], activation=cfg["pbld_activation"],
        )
        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total_dim   = num_emb_dim + cat_emb_dim
        hidden_dims = cfg["hidden_dims"]
        act = cfg["activation"]

        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))
        self._dropout_modules = []
        in_dim = total_dim
        for i, out_dim_h in enumerate(hidden_dims):
            linear = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=out_dim_h)
            if i == 0:
                self.first_linear = linear
            drop = nn.Dropout(cfg["dropout"])
            self._dropout_modules.append(drop)
            layers += [linear, act(), drop]
            in_dim = out_dim_h

        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        combined = torch.cat([x_num, x_cat], dim=2)
        x = self.hidden(combined)
        x = self.output_layer(x)
        return F.softmax(x, dim=2)


# ── Schedule helpers ──────────────────────────────────────────────────────────
def apply_schedule(init_value, progress, sched, flat_ratio=0.3):
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")


def get_parameter_groups(model, p):
    first_linear_weight_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    LR = p["lr"]
    WD = p["weight_decay"]
    return [
        {"params": scale_p,   "lr": LR * p["lr_scale_mult"],         "weight_decay": WD * p["wd_scale_mult"],         "group": "scale"},
        {"params": pbld_p,    "lr": LR * p["pbld_lr_factor"],        "weight_decay": WD,                              "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay": WD * p["first_layer_wd_factor"], "group": "first_w"},
        {"params": other_w_p, "lr": LR,                              "weight_decay": WD,                              "group": "other_w"},
        {"params": bias_p,    "lr": LR * p["lr_bias_mult"],          "weight_decay": WD * p["wd_bias_mult"],          "group": "bias"},
    ]


def smooth_ce_loss(y_true, y_pred, ls=0.0, class_weights=None):
    n_classes = y_pred.size(1)
    y_smooth  = torch.full_like(y_pred, ls / n_classes)
    y_smooth.scatter_(1, y_true.unsqueeze(1), 1.0 - ls + ls / n_classes)
    per_sample_loss = -(y_smooth * torch.log(y_pred.clamp(1e-15, 1))).sum(dim=1)
    if class_weights is not None:
        sample_weights = class_weights[y_true]
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum()
    return per_sample_loss.mean()


# ── Sklearn-compatible wrapper ────────────────────────────────────────────────
class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kwargs):
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train, y_train, X_val, y_val,
            cat_col_names=None, ckpt_path="realmlp_ckpt.pth", X_test=None):
        p   = self.params
        dev = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        X_tr_num  = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat  = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr = np.asarray(y_train)
        y_v  = np.asarray(y_val)

        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num  = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        if cat_col_names:
            all_cat  = [X_tr_cat, X_val_cat]
            if X_test is not None:
                all_cat.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims = (np.concatenate(all_cat, axis=0).max(axis=0) + 1).tolist()
        else:
            cat_dims = []
        self.cat_dims_ = cat_dims

        if cat_dims:
            cat_max   = np.array(cat_dims) - 1
            X_tr_cat  = np.clip(X_tr_cat,  0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        classes       = np.unique(y_tr)
        self.classes_ = classes
        weights_np    = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weights = torch.as_tensor(weights_np, dtype=torch.float32, device=dev)

        n_classes   = len(classes)
        self.model_ = RealMLP(
            output_dim=n_classes, cat_dims=cat_dims,
            n_numerical=X_tr_num.shape[1], cfg=p,
        ).to(dev)

        param_groups = get_parameter_groups(self.model_, p)
        for g in param_groups:
            g["lr_base"] = g["lr"]
        optimizer = torch.optim.AdamW(param_groups, betas=(p["mom"], p["sq_mom"]))

        Xtn = torch.as_tensor(X_tr_num,  dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat,  dtype=torch.long,    device=dev)
        ytt = torch.as_tensor(y_tr,      dtype=torch.long,    device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long,    device=dev)

        n_ens       = p["n_ens"]
        train_bs    = p["train_bs"]
        eval_bs     = p["eval_bs"]
        epochs      = p["epochs"]
        lr_sched    = p["lr_sched"]
        flat_ratio  = p["flat_ratio"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))

        best_score     = -np.inf
        best_epoch     = 0
        best_val_probs = None
        self.ckpt_path_ = ckpt_path

        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress  = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start : start + train_bs]
                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)
                optimizer.zero_grad()
                y_pred = self.model_(Xtn[idx_batch], Xtc[idx_batch])
                ls_val   = apply_schedule(p["ls_eps"],  progress, p["ls_eps_sched"],  flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"],  flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val
                loss = smooth_ce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1, n_classes),
                    ls=ls_val, class_weights=class_weights,
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()
            np.random.shuffle(train_order)

            self.model_.eval()
            with torch.no_grad():
                val_probs = np.concatenate([
                    self.model_(Xvn[s:s+eval_bs], Xvc[s:s+eval_bs])
                        .mean(dim=1).cpu().numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)

            epoch_score = balanced_accuracy_score(y_v, np.argmax(val_probs, axis=1))
            improved    = epoch_score > best_score
            if improved:
                best_score     = epoch_score
                best_epoch     = epoch + 1
                best_val_probs = val_probs.copy()
                torch.save(self.model_.state_dict(), ckpt_path)

            if verbose >= 2:
                print(f"  epoch {epoch+1}/{epochs}  score={epoch_score:.5f}  best={best_score:.5f}" + (" ✓" if improved else ""))

            if p["use_early_stopping"]:
                patience = (best_epoch * p["early_stopping_multiplicative_patience"]
                            + p["early_stopping_additive_patience"])
                if (epoch + 1) > patience:
                    if verbose >= 1:
                        print(f"  Early stopping at epoch {epoch+1} (best epoch {best_epoch})")
                    break

        self.model_.load_state_dict(torch.load(ckpt_path))
        self.best_score_     = best_score
        self.best_val_probs_ = best_val_probs
        self._dev            = dev
        if verbose >= 1:
            print(f"  → best score: {best_score:.5f}  (epoch {best_epoch})")
        return self

    def predict_proba(self, X):
        eval_bs = self.params["eval_bs"]
        X_num = self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        X_cat = np.clip(X[self.cat_col_names_].values.astype(np.int64), 0, np.array(self.cat_dims_) - 1)
        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long,    device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([
                self.model_(Xn[s:s+eval_bs], Xc[s:s+eval_bs])
                    .mean(dim=1).cpu().numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

In [8]:
#　RealMLP　本体　宣言

CONFIG = {
    # モデル構造
    "n_ens":         8,
    "embed_dim":     8,
    "onehot_thresh": 8,
    "hidden_dims":   [512, 512, 512],
    "dropout":       0.06,
    "p_drop_sched":  "expm4t",
    "activation":    nn.SiLU,
    "add_front_scale": True,

    # PBLD（数値の周期埋め込み）
    "pbld_hidden_dim": 20,
    "pbld_out_dim":    5,
    "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU,
    "pbld_lr_factor":  0.093,

    # Optimizer
    "lr":               0.01,
    "mom":              0.9,
    "sq_mom":           0.98,
    "lr_sched":        "flat_cos",
    "flat_ratio":       0.3,
    "first_layer_lr_factor": 1.0,
    "first_layer_wd_factor": 0.1,
    "lr_scale_mult":    10.0,
    "lr_bias_mult":     0.1,
    "weight_decay":     0.013,
    "wd_scale_mult":    0.1,
    "wd_bias_mult":     0.5,
    "grad_clip":        1.0,

    # ラベルスムージング
    "ls_eps":       0.04,
    "ls_eps_sched": "cos",

    # 前処理
    "tfms": ["median_center", "robust_scale"],

    # 学習ループ
    "epochs":    10,
    "train_bs":  256,
    "eval_bs":   10240,
    "verbosity": 2,

    # Early stopping
    "use_early_stopping":                     False,
    "early_stopping_additive_patience":       10,
    "early_stopping_multiplicative_patience": 1,

    # デバイス
    "device":       "cuda",
    "random_state": 42,
}

feature_cols = [
    "alpha", "delta", "u", "g", "r", "i", "z", "redshift",
    "u_g", "u_r", "u_i", "g_r", "r_i", "i_z",
    "alpha_sin", "alpha_cos", "redshift_log",
    "spectral_type", "galaxy_population",
    "alpha_cat", "delta_cat", "redshift_cat",
    "u_cat", "g_cat", "r_cat", "i_cat", "z_cat",
    "alpha_delta_combo", "u_z_combo",
    "delta_100bin", "delta_500bin",
    "redshift_50bin", "redshift_100bin",
    "alpha_100bin",
]

X      = train[feature_cols]
y      = train["class_encoded"]
X_test = test[feature_cols]

def seed_everything(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

def objective_realmlp(trial):
    cfg = {
        # ── モデル構造 ──
        "n_ens":      trial.suggest_int("n_ens", 4, 16),
        "hidden_dims": [
            trial.suggest_int("hidden_dim", 128, 1024, step=128)
        ] * trial.suggest_int("n_layers", 2, 5),
        "dropout":    trial.suggest_float("dropout", 0.0, 0.2),
        "activation": nn.SiLU,
        "embed_dim":  trial.suggest_int("embed_dim", 4, 16),
        "onehot_thresh": 8,
        "add_front_scale": True,

        # ── PBLD ──
        "pbld_hidden_dim": trial.suggest_int("pbld_hidden_dim", 8, 32),
        "pbld_out_dim":    trial.suggest_int("pbld_out_dim", 3, 8),
        "pbld_freq_scale": trial.suggest_float("pbld_freq_scale", 1.0, 10.0),
        "pbld_activation": nn.PReLU,
        "pbld_lr_factor":  trial.suggest_float("pbld_lr_factor", 0.01, 0.2),

        # ── Optimizer ──
        "lr":           trial.suggest_float("lr", 1e-3, 5e-2, log=True),
        "mom":          0.9,
        "sq_mom":       0.98,
        "lr_sched":     "flat_cos",
        "flat_ratio":   trial.suggest_float("flat_ratio", 0.1, 0.5),
        "weight_decay": trial.suggest_float("weight_decay", 1e-4, 0.1, log=True),
        "first_layer_lr_factor": 1.0,
        "first_layer_wd_factor": 0.1,
        "lr_scale_mult": 10.0,
        "lr_bias_mult":  0.1,
        "wd_scale_mult": 0.1,
        "wd_bias_mult":  0.5,
        "grad_clip":     1.0,

        # ── Label Smoothing ──
        "ls_eps":       trial.suggest_float("ls_eps", 0.01, 0.1),
        "ls_eps_sched": "cos",

        # ── Dropout Schedule ──
        "p_drop_sched": "expm4t",

        # ── 前処理 ──
        "tfms": ["median_center", "robust_scale"],

        # ── 学習ループ ──
        "epochs":   trial.suggest_int("epochs", 8, 20),
        "train_bs": 256,
        "eval_bs":  10240,
        "verbosity": 0,  # ← Optuna中はログ非表示

        # ── Early stopping ──
        "use_early_stopping": False,
        "early_stopping_additive_patience":       10,
        "early_stopping_multiplicative_patience": 1,

        # ── Device ──
        "device":       "cuda",
        "random_state": 42,
    }

    # ===== 1foldだけで高速評価 =====
    tr_idx, val_idx = next(iter(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y)
    ))

    X_tr  = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()
    y_tr  = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    # Target Encoding
    encoder = TargetEncoder(cv=5, smooth='auto', shuffle=True, random_state=42)
    te_cols    = ["alpha_delta_combo", "u_z_combo"]
    tr_enc  = encoder.fit_transform(X_tr[te_cols], y_tr)
    val_enc = encoder.transform(X_val[te_cols])
    tst_enc = encoder.transform(X_tst[te_cols])

    te_names = [f"{col}_TE_cls{c}" for col in te_cols for c in range(3)]
    X_tr[te_names]  = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    try:
        model = RealMLP_TD_Classifier(**cfg)
        model.fit(
            X_tr, y_tr,
            X_val, y_val,
            cat_col_names=cat_cols,
            ckpt_path=f"realmlp_optuna_trial{trial.number}.pth",
        )
        return model.best_score_
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return 0.0

# ===== Optuna実行 =====
study_mlp = optuna.create_study(direction="maximize")
study_mlp.optimize(objective_realmlp, n_trials=30, show_progress_bar=True)

print(f"\nBest OOF Score: {study_mlp.best_value:.5f}")
print(f"Best Params:\n{study_mlp.best_params}")

device: cuda


  0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_2608/1088106106.py:295: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  ytt = torch.as_tensor(y_tr,      dtype=torch.long,    device=dev)



Best OOF Score: 0.96980
Best Params:
{'n_ens': 12, 'hidden_dim': 512, 'n_layers': 2, 'dropout': 0.10098735032634398, 'embed_dim': 6, 'pbld_hidden_dim': 24, 'pbld_out_dim': 4, 'pbld_freq_scale': 8.102895749115502, 'pbld_lr_factor': 0.05252255959065255, 'lr': 0.003654758364470636, 'flat_ratio': 0.11375350656712123, 'weight_decay': 0.013865520547248794, 'ls_eps': 0.09878481651554702, 'epochs': 17}


In [ ]:
# 検証段階

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


oof_preds_mlp   = np.zeros((len(X), 3))
oof_preds_cat   = np.zeros((len(X), 3))  # ← 追加
test_preds_mlp  = np.zeros((len(X_test), 3))
test_preds_cat  = np.zeros((len(X_test), 3))  # ← 追加

X_tst_base = X_test.copy()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'#'*16}")
    print(f"### Fold {fold}/5")
    print(f"{'#'*16}")

    X_tr  = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_tst_base.copy()
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # ===== Target Encoding =====
    encoder = TargetEncoder(cv=5, smooth='auto', shuffle=True, random_state=42)
    te_cols    = ["alpha_delta_combo", "u_z_combo"]
    tr_enc  = encoder.fit_transform(X_tr[te_cols], y_tr)
    val_enc = encoder.transform(X_val[te_cols])
    tst_enc = encoder.transform(X_tst[te_cols])

    te_names = [f"{col}_TE_cls{c}" for col in te_cols for c in range(3)]
    X_tr[te_names]  = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    
    # ===== CatBoost =====
    print("\n[CatBoost]")
    X_tr_cat  = X_tr.copy()
    X_val_cat = X_val.copy()
    X_tst_cat = X_tst.copy()

    for df in [X_tr_cat, X_val_cat, X_tst_cat]:
        for col in df.select_dtypes(include="category").columns:
            df[col] = df[col].astype(int)

    cat_col_indices_fold = [X_tr_cat.columns.get_loc(c) for c in cat_cols if c in X_tr_cat.columns]

    cat_model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        random_seed=42,
        early_stopping_rounds=50,
        verbose=100,
        cat_features=cat_col_indices_fold,
    )
    cat_model.fit(
        X_tr_cat, y_tr,
        eval_set=(X_val_cat, y_val),
    )
    oof_preds_cat[val_idx]  = cat_model.predict_proba(X_val_cat)
    test_preds_cat          += cat_model.predict_proba(X_tst_cat) / 5

    # ===== RealMLP =====
    print("\n[RealMLP]")
    mlp_model = RealMLP_TD_Classifier(**CONFIG)
    mlp_model.fit(
        X_tr, y_tr,
        X_val, y_val,
        cat_col_names=cat_cols,
        ckpt_path=f"realmlp_fold{fold}.pth",
    )
    oof_preds_mlp[val_idx]  = mlp_model.best_val_probs_
    test_preds_mlp         += mlp_model.predict_proba(X_tst) / 5

# ===== OOF スコア確認 =====
mlp_oof  = balanced_accuracy_score(y, np.argmax(oof_preds_mlp,  axis=1))
cat_oof  = balanced_accuracy_score(y, np.argmax(oof_preds_cat,  axis=1))

print(f"\n{'='*30}")
print(f"RealMLP  OOF: {mlp_oof:.5f}")
print(f"CatBoost OOF: {cat_oof:.5f}")

# ===== 重み探索 =====
for mlp_w in np.arange(0, 1.1, 0.1):
    cat_w = round(1.0 - mlp_w, 1)
    oof_ens = mlp_w * oof_preds_mlp + cat_w * oof_preds_cat
    score   = balanced_accuracy_score(y, np.argmax(oof_ens, axis=1))
    print(f"MLP={mlp_w:.1f} CAT={cat_w:.1f} → OOF: {score:.5f}")

# ===== 最適重みで提出 =====
# 重み探索結果を見てから設定して！
MLP_W = 0.8
CAT_W = 0.2

test_ensemble = MLP_W * test_preds_mlp + CAT_W * test_preds_cat
pred_labels   = le.inverse_transform(np.argmax(test_ensemble, axis=1))
submission    = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission.csv", index=False)
print(submission["class"].value_counts())


################
### Fold 1/5
################

[CatBoost]
0:	learn: 0.9369572	test: 0.9373257	best: 0.9373257 (0)	total: 694ms	remaining: 11m 33s
100:	learn: 0.9612278	test: 0.9617650	best: 0.9617650 (100)	total: 55.4s	remaining: 8m 13s
200:	learn: 0.9656359	test: 0.9659565	best: 0.9659565 (200)	total: 1m 50s	remaining: 7m 19s
300:	learn: 0.9675130	test: 0.9677492	best: 0.9677492 (300)	total: 2m 43s	remaining: 6m 20s
400:	learn: 0.9685912	test: 0.9684853	best: 0.9684853 (400)	total: 3m 38s	remaining: 5m 26s
500:	learn: 0.9692710	test: 0.9691695	best: 0.9691955 (498)	total: 4m 35s	remaining: 4m 34s
600:	learn: 0.9697582	test: 0.9695938	best: 0.9695938 (600)	total: 5m 31s	remaining: 3m 40s
700:	learn: 0.9702193	test: 0.9698969	best: 0.9699143 (691)	total: 6m 26s	remaining: 2m 44s
800:	learn: 0.9704640	test: 0.9701741	best: 0.9702174 (797)	total: 7m 22s	remaining: 1m 49s
900:	learn: 0.9707541	test: 0.9703300	best: 0.9704252 (858)	total: 8m 14s	remaining: 54.3s
Stopped by overfitting det

In [15]:
# ===== 最適重みで提出 =====
# 重み探索結果を見てから設定して！
MLP_W = 0.9
CAT_W = 0.1

test_ensemble = MLP_W * test_preds_mlp + CAT_W * test_preds_cat
pred_labels   = le.inverse_transform(np.argmax(test_ensemble, axis=1))
submission    = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission.csv", index=False)
print(submission["class"].value_counts())

class
GALAXY    157891
QSO        51269
STAR       38275
Name: count, dtype: int64


In [14]:
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler

# ===== メタ特徴量を作成 =====
train_meta = np.hstack([oof_preds_mlp,oof_preds_cat])
test_meta  = np.hstack([test_preds_mlp, test_preds_cat])

print("train_meta shape:", train_meta.shape)  # (N, 12)
print("test_meta shape: ", test_meta.shape)   # (N, 12)

# ===== スケーリング =====
scaler = StandardScaler()
train_meta_scaled = scaler.fit_transform(train_meta)
test_meta_scaled  = scaler.transform(test_meta)

# ===== alphaを色々試す =====
best_alpha = None
best_score = 0

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = RidgeClassifier(alpha=alpha)
    ridge.fit(train_meta_scaled, y)
    score = balanced_accuracy_score(y, ridge.predict(train_meta_scaled))
    print(f"alpha={alpha:.3f} → OOF: {score:.5f}")
    if score > best_score:
        best_score = score
        best_alpha = alpha

print(f"\nBest alpha: {best_alpha} → OOF: {best_score:.5f}")

# ===== ベストalphaで提出ファイル作成 =====
best_ridge = RidgeClassifier(alpha=best_alpha)
best_ridge.fit(train_meta_scaled, y)

pred_labels = le.inverse_transform(best_ridge.predict(test_meta_scaled))
submission  = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission_stacking.csv", index=False)
print(submission["class"].value_counts())

train_meta shape: (577347, 6)
test_meta shape:  (247435, 6)
alpha=0.010 → OOF: 0.96216
alpha=0.100 → OOF: 0.96216
alpha=1.000 → OOF: 0.96216
alpha=10.000 → OOF: 0.96216
alpha=100.000 → OOF: 0.96219

Best alpha: 100.0 → OOF: 0.96219
class
GALAXY    161328
QSO        50316
STAR       35791
Name: count, dtype: int64


In [10]:
"""
#　スタッキング
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ===== メタ特徴量を作成 =====
# OOFの予測値を横に並べる (N, 6)
train_meta = np.hstack([oof_preds_lgbm, oof_preds_mlp])
test_meta  = np.hstack([test_preds_lgbm, test_preds_mlp])

print("train_meta shape:", train_meta.shape)  # (N, 6)
print("test_meta shape: ", test_meta.shape)   # (N, 6)

# ===== スケーリング =====
scaler = StandardScaler()
train_meta_scaled = scaler.fit_transform(train_meta)
test_meta_scaled  = scaler.transform(test_meta)

# ===== ロジスティック回帰で学習 =====
# Cを色々試す
for C in [0.01, 0.1, 1.0, 10.0]:
    meta_model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    meta_model.fit(train_meta_scaled, y)
    
    oof_meta = meta_model.predict_proba(train_meta_scaled)
    score    = balanced_accuracy_score(y, np.argmax(oof_meta, axis=1))
    print(f"C={C:.3f} → OOF: {score:.5f}")
"""

'\n#\u3000スタッキング\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\n\n# ===== メタ特徴量を作成 =====\n# OOFの予測値を横に並べる (N, 6)\ntrain_meta = np.hstack([oof_preds_lgbm, oof_preds_mlp])\ntest_meta  = np.hstack([test_preds_lgbm, test_preds_mlp])\n\nprint("train_meta shape:", train_meta.shape)  # (N, 6)\nprint("test_meta shape: ", test_meta.shape)   # (N, 6)\n\n# ===== スケーリング =====\nscaler = StandardScaler()\ntrain_meta_scaled = scaler.fit_transform(train_meta)\ntest_meta_scaled  = scaler.transform(test_meta)\n\n# ===== ロジスティック回帰で学習 =====\n# Cを色々試す\nfor C in [0.01, 0.1, 1.0, 10.0]:\n    meta_model = LogisticRegression(C=C, max_iter=1000, random_state=42)\n    meta_model.fit(train_meta_scaled, y)\n\n    oof_meta = meta_model.predict_proba(train_meta_scaled)\n    score    = balanced_accuracy_score(y, np.argmax(oof_meta, axis=1))\n    print(f"C={C:.3f} → OOF: {score:.5f}")\n'

In [11]:
"""
# スタッキングベストパラメータで実装
# ベストCで再学習（例: C=1.0が良かった場合）
best_meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
best_meta_model.fit(train_meta_scaled, y)

# 予測
test_meta_proba = best_meta_model.predict_proba(test_meta_scaled)
pred_labels     = le.inverse_transform(np.argmax(test_meta_proba, axis=1))

submission = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission_stacking.csv", index=False)
print(submission["class"].value_counts())
"""

'\n# スタッキングベストパラメータで実装\n# ベストCで再学習（例: C=1.0が良かった場合）\nbest_meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)\nbest_meta_model.fit(train_meta_scaled, y)\n\n# 予測\ntest_meta_proba = best_meta_model.predict_proba(test_meta_scaled)\npred_labels     = le.inverse_transform(np.argmax(test_meta_proba, axis=1))\n\nsubmission = pd.DataFrame({"id": test["id"], "class": pred_labels})\nsubmission.to_csv("submission_stacking.csv", index=False)\nprint(submission["class"].value_counts())\n'